# 特徴量エンジニアリング

欠損補完、カテゴリ変数エンコーディング、対数変換など

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append("..")
from src.features.build_features import FeatureBuilder

train_raw = pd.read_csv("../data/raw/train.csv")
test_raw = pd.read_csv("../data/raw/test.csv")
train_raw.shape, test_raw.shape

((1460, 81), (1459, 80))

## 1. 外れ値除外(01_eda.ipynbで確認したGrLivArea外れ値)

In [2]:
outlier_mask = (train_raw["GrLivArea"] > 4000) & (train_raw["SalePrice"] < 300000)
print(f"除外する外れ値: {outlier_mask.sum()}件")
train_raw = train_raw[~outlier_mask].reset_index(drop=True)
train_raw.shape

除外する外れ値: 2件


(1458, 81)

## 2. 特徴量ビルド(train統計量でfit → train/testにtransform)

In [3]:
sale_price = train_raw["SalePrice"]
X_train_raw = train_raw.drop(columns=["SalePrice"])
X_test_raw = test_raw.copy()

fb = FeatureBuilder()
X_train = fb.fit_transform(X_train_raw)
X_test = fb.transform(X_test_raw)

X_train.shape, X_test.shape

((1458, 266), (1459, 266))

In [4]:
# 欠損が残っていないか確認
assert X_train.isnull().sum().sum() == 0, "X_trainに欠損が残っています"
assert X_test.isnull().sum().sum() == 0, "X_testに欠損が残っています"
print("欠損なしを確認")

欠損なしを確認


## 3. data/processed/ へ保存

`src/models/train.py` は `train_features.csv` から `SalePrice` を、
`src/models/predict.py` は `test_features.csv` から `Id` を読む前提。

In [5]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

train_out = X_train.drop(columns=["Id"]).copy()
train_out["SalePrice"] = sale_price.values
train_out.to_csv(processed_dir / "train_features.csv", index=False)

test_out = X_test.copy()  # Idはpredict.pyがpopして提出csvに使う
test_out.to_csv(processed_dir / "test_features.csv", index=False)

print(f"train_features.csv: {train_out.shape}")
print(f"test_features.csv: {test_out.shape}")

train_features.csv: (1458, 266)
test_features.csv: (1459, 266)
